In [3]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

Layer Normalization

In [4]:
import torch
import torch.nn as nn

In [5]:
class LayerNorm(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(out_dim))
        self.shift = nn.Parameter(torch.zeros(out_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim = True)
        var = x.var(dim=-1, keepdim = True, unbiased = False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)

        return self.scale * x_norm + self.shift

In [6]:
batch = torch.randn(2,5)
batch

layer = nn.Sequential(nn.Linear(5,6), nn.ReLU())
x = layer(batch)

In [7]:
torch.set_printoptions(sci_mode=False)

In [8]:
ln = LayerNorm(6)
out = ln(x)
out.mean(dim = -1)

tensor([0.0000, 0.0000], grad_fn=<MeanBackward1>)

GeLU Activation Layer

In [9]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self,x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0/torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
            ))

In [10]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [11]:
inp = torch.rand(2, 3, 768)
ffn = FeedForward(GPT_CONFIG_124M)
out = ffn(inp)

Shortcut Connections

In [12]:
class SampleDeepNeuralNetwork(nn.Module):

    def __init__(self, layer_sizes):
        super().__init__()

        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU()),
        ])

    def forward(self, x):
        for layer in self.layers:
            out = layer(x)
            #x = out without shortcut
            x = x + out #shortcut
        return x  
    
def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([0.])
    loss = nn.MSELoss()
    loss = loss(output, target)
    loss.backward()
    for name, param in model.named_parameters():
        if "weight" in name:
            print(name, "has gradient mean: ", param.grad.abs().mean().item())


In [13]:
layer_sizes = [3,3,3,3,3,1]
dnn = SampleDeepNeuralNetwork(layer_sizes)

torch.manual_seed(123)
inp = torch.tensor([[1., 0., -1.]])
print_gradients(dnn, inp)


layers.0.0.weight has gradient mean:  0.43687012791633606
layers.1.0.weight has gradient mean:  0.5457799434661865
layers.2.0.weight has gradient mean:  0.7667669653892517
layers.3.0.weight has gradient mean:  0.8417879939079285
layers.4.0.weight has gradient mean:  0.9640681147575378


c:\Users\Jennifer\anaconda3\envs\LLMs\lib\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 3])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Transformer Block

In [14]:
from package import MultiHeadAttention

class TransformerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.layer_norm1 = LayerNorm(cfg["emb_dim"])
        self.layer_norm2 = LayerNorm(cfg["emb_dim"])
        self.mha = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"], cfg["context_length"], cfg["drop_rate"], cfg["n_heads"], cfg["qkv_bias"])
        self.dropout = nn.Dropout(cfg["drop_rate"])
        self.feed_forward = FeedForward(cfg)

    def forward(self, x):
        shortcut = x
        x = self.layer_norm1(x)
        x = self.mha(x)
        x = self.dropout(x)
        x = x + shortcut

        x = self.layer_norm2(x)
        x = self.feed_forward(x)
        x = self.dropout(x)
        x = x + shortcut

        return x


In [15]:
inp = torch.rand(2,3,768)
tb = TransformerBlock(GPT_CONFIG_124M)
out = tb(inp)
out.shape

torch.Size([2, 3, 768])

Simple GPT Model

In [45]:
import tiktoken

In [33]:
class GPTModel(nn.Module):
    
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding_layer = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_embedding_layer = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.dropout_layer = nn.Dropout(cfg["drop_rate"])
        self.trans_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.layer_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(in_features=cfg["emb_dim"], out_features=cfg["vocab_size"], bias=False)


    def forward(self, x):
        batch_size, seq_len = x.shape
        token_embeddings = self.token_embedding_layer(x)
        pos_embeddings = self.pos_embedding_layer(torch.arange(seq_len, device=x.device))
        input = token_embeddings + pos_embeddings

        out = self.dropout_layer(input)
        out = self.trans_blocks(out)
        out = self.layer_norm(out)
        logits = self.out_head(out)

        return logits
    

In [35]:
inp = torch.randint(0, 50256, (2,4))
model = GPTModel(GPT_CONFIG_124M)
out = model(inp)
out.shape

torch.Size([2, 4, 50257])

In [56]:
def generate_text(model, tokens, tokenizer, context_length):
    tokens = tokens[:, -context_length:]
    with torch.no_grad():
        out = model(tokens)

    logits = out[:, -1, :]
    max_ind = torch.argmax(logits, dim=-1, keepdim=True)
    ind = torch.cat((tokens, max_ind), dim=1)

    return ind

In [66]:
tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello I am a"
encoded = torch.tensor(tokenizer.encode(text)).unsqueeze(0)
encoded_out = generate_text(model, encoded, tokenizer, GPT_CONFIG_124M["context_length"] )

tokenizer.decode(encoded_out.squeeze(0).tolist())

'Hello I am aMail'